In [1]:
# conda env at3d_pure

import xarray as xr
import numpy as np
import at3d
from collections import OrderedDict
import matplotlib.pyplot as plt

In [2]:
wavelength = 0.67
hsrl_file = '../data/CAMP2Ex_HSRL_20191001_R0.nc'

rte_grid_title = 'default'
rte_grid_file = '../data/RTG_{}.nc'.format(rte_grid_title)
rte_grid = xr.load_dataset(rte_grid_file)

out_file = '../data/AEROSOL_CAMP2Ex_HSRL_20191001_RTG_{}.nc'.format(rte_grid_title)

In [3]:
# Aerosols
"""
OPAC Sea salt (acc. mode) at 80% RH.
r_min = 0.009 (um)
r_max = 39.9 (um)
r_mod = 0.416 (um)
sigma = 2.03 (unitless)
rho = 1.15 (g m^-3)
m(lamda = 0.65 um) = 1.351 - 1j*1.97e-8

Notes
-----
Rho comes from Jesse's size_distr.cfg in located in 'AT3D/data/OPAC/aerosol' -- 
I can't find this parameter in the refs below.

I believe there is a couple of typos in Hess et al. (1998). First, I believe
there is a missing negative sign in front of the 1/2 within the exponential in
equation 3d. Second, I believe they are reporting the median radius - not the
mode radius as they describe in the text. This makes more sense when plotting
the distribution (eq. 3d).

(1/3/25) Aerosols above domain are not accounted for

References
----------
https://cds-espri.ipsl.upmc.fr/etherTypo/fileadmin/files/GEISA/hess-etal2.pdf
https://geisa.aeris-data.fr/opac/
"""
aerosol_mie_table = OrderedDict()
aerosol_mie_table[wavelength] = at3d.mie.get_mono_table(
    'Aerosol',
    (wavelength,wavelength),
    max_integration_radius=39.9,
    minimum_effective_radius=0.009,
    refractive_index = 1.351 - 1j*1.97e-8,
    relative_dir='../../mie_tables',
    verbose=False
)
# aerosol_mie_table[wavelength].to_netcdf('../../mie_tables/aerosol_mie_table.nc')

aerosol_optical_prop_gen = at3d.medium.OpticalPropertyGenerator(
    'aerosol',
    aerosol_mie_table,
    at3d.size_distribution.lognormal,
    particle_density=1.15,
    maxnphase=None,
    interpolation_mode='exact',
    density_normalization='geometric_extinction',
    reff=np.linspace(0.005, 8, 20),
    veff=np.linspace(0.1, 0.9, 5)
)

r_median = 0.416
S = 2.03
r_mode = np.exp(np.log(r_median) - np.log(S)**2)
reff = r_mode * np.exp((5/2)*(np.log(S)**2))
veff = np.exp(np.log(S)**2) - 1

aerosol = rte_grid.copy(deep=True)
aerosol['density'] = (['x','y','z'], np.zeros((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z'])))
aerosol['reff'] = (['x','y','z'], np.ones(aerosol.density.shape) * reff)
aerosol['veff'] = (['x','y','z'], np.ones(aerosol.density.shape) * veff)
aerosol_optical_properties = aerosol_optical_prop_gen(aerosol)

hsrl = xr.load_dataset(hsrl_file)
hsrl = hsrl.isel(gps_time=slice(1400,2122))
hsrl['altitude'] = hsrl.altitude / 1000

## COMPENSATE FOR AEROSOLS ABOVE DOMAIN
aerosol_profile_exact = hsrl.mean('gps_time', skipna=True).ext670_532_355.copy(deep=True)
aerosol_profile_exact[np.isnan(aerosol_profile_exact.data)] = 0
tau_exact = aerosol_profile_exact.integrate('altitude').data

aerosol_profile_coarse = aerosol_profile_exact.interp(altitude=rte_grid.z.data).copy(deep=True)
aerosol_profile_minus1 = aerosol_profile_coarse.copy(deep=True).isel(altitude=slice(0,-1))
tau_minus1 = aerosol_profile_minus1.integrate('altitude').data

del_tau = tau_exact - tau_minus1
h = aerosol_profile_coarse.altitude.data[-1] - aerosol_profile_coarse.altitude.data[-2]
xmax = aerosol_profile_minus1.isel(altitude=-1).data
compensation_extinction = ((2*del_tau)/h) - xmax
aerosol_profile_comp = aerosol_profile_coarse.copy(deep=True)
aerosol_profile_comp[-1] = compensation_extinction

aerosol_extinction = np.tile(aerosol_profile_comp.data, (aerosol.dims['x'], aerosol.dims['y'], 1))
aerosol_optical_properties[wavelength]['extinction'] = (['x','y','z'], aerosol_extinction)

aerosol_optical_properties[wavelength].to_netcdf(out_file)

In [4]:
# # PLOT CORRECTED EXTINCTION
# f, ax = plt.subplots(1, 1, figsize=(8,6))
# aerosol_profile_exact.plot(y='altitude', marker='.', ax=ax)
# aerosol_profile_coarse.plot(y='altitude', marker='.', ax=ax)
# aerosol_profile_minus1.plot(y='altitude', marker='.', ax=ax)
# aerosol_optical_properties[wavelength].isel(x=4,y=4).extinction.plot(y='z', marker='.', ax=ax)
# ax.grid()
# tau_exact = aerosol_profile_exact.integrate('altitude').data
# tau = aerosol_profile_coarse.integrate('altitude').data
# tau_minus1 = aerosol_profile_minus1.integrate('altitude').data
# tau_comp = aerosol_profile_comp.integrate('altitude').data
# print('tau_exact:  {:.5f}'.format(tau_exact))
# print('tau:        {:.5f}'.format(tau))
# print('tau minus1: {:.5f}'.format(tau_minus1))
# print('tau comp:   {:.5f}'.format(tau_comp))